In [0]:
%pip install /Workspace/Users/neil.braun@mirakl.com/.bundle/fast-gnn-benchmark/dev/files
dbutils.library.restartPython()

In [0]:
from fast_gnn_benchmark.trainer import load_model_from_checkpoint
from fast_gnn_benchmark.data.dataset.coview_mdm import CoViewMDMDataset

import os
import copy
import time
import numpy as np
import torch
import boto3, json, io
import pyspark.sql.functions as F
from pyspark.sql import DataFrame, Row
from IPython.display import display, HTML

In [0]:
from datetime import datetime

# Dossier propre au prototype: gae_gcn_coview_mdm_prototype.yml y ecrit ses checkpoints, pour ne pas
# polluer coview-mdm-128 que model_test.ipynb (non-prototype) lit toujours.
ckpt_dir = "/dbfs/tmp/nbraun/checkpoints/coview-mdm-prototype"

files = [
    (os.path.getmtime(os.path.join(ckpt_dir, f)), f)
    for f in os.listdir(ckpt_dir)
    if f.endswith(".ckpt")
]

for mtime, fname in sorted(files, reverse=True):
    print(f"{datetime.fromtimestamp(mtime):%Y-%m-%d %H:%M:%S}  {fname}")

In [0]:
from fast_gnn_benchmark.models.link_prediction import LinkPredictionModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# save_top_k=1 sur val/mrr_trigger: le dossier ne contient que le checkpoint du meilleur epoch. On le
# resout dynamiquement, son nom dependant de l'epoch atteint -- pas de nom de fichier en dur ici.
assert files, f"aucun .ckpt dans {ckpt_dir}: lancer l'entrainement avec gae_gcn_coview_mdm_prototype.yml"
checkpoint_path = os.path.join(ckpt_dir, sorted(files, reverse=True)[0][1])
print(f"checkpoint: {checkpoint_path}")

raw_ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

model = LinkPredictionModel.load_from_checkpoint(checkpoint_path, map_location=device, weights_only=False)
model.eval()
model.to(device)

print("epoch:", raw_ckpt["epoch"])
print("global_step:", raw_ckpt["global_step"])
assert f"epoch={raw_ckpt['epoch']}-step={raw_ckpt['global_step']}" in checkpoint_path

print(model.hparams.model_parameters)

for cb_state in raw_ckpt["callbacks"].values():
    if "best_model_score" in cb_state:
        print("best_model_score (val/mrr_trigger):", cb_state["best_model_score"])

print("nb params:", sum(p.numel() for p in model.parameters()))

## Chargement du dataset et des mappings prototype

In [0]:
BUCKET = "mirakl-data-science-tmp2"
PREFIX = "nbraun/datasets/coview-mdm"

# Artefacts prototype: exec2code y couvre toutes les executions du split, donc les exec_code
# diffèrent de ceux du pipeline principal. Ne jamais melanger avec data.pt / exec_mappings.json.
dataset = CoViewMDMDataset(bucket=BUCKET, s3_key=f"{PREFIX}/data_prototype.pt")

s3 = boto3.client("s3")

node2idx_raw = json.loads(
    s3.get_object(Bucket=BUCKET, Key=f"{PREFIX}/node2idx_prototype.json")["Body"].read()
)
node2idx = {int(k): v for k, v in node2idx_raw.items()}
idx2node = {v: k for k, v in node2idx.items()}

exec_mappings = json.loads(
    s3.get_object(Bucket=BUCKET, Key=f"{PREFIX}/exec_mappings_prototype.json")["Body"].read()
)

print(f"num_nodes: {dataset.num_nodes}")
print(f"node2idx entries: {len(node2idx)}")
for split in ["train", "val", "test"]:
    print(f"{split}: {len(exec_mappings[split]['code2exec'])} executions")

## Tables prototype

`prod_results_val_prototype` porte les deux variantes du retour prod pour chaque trigger : `_display` (les 12 premiers dans l'ordre d'affichage) et `_relevance` (les 12 premiers par `relevanceScore`). C'est aussi elle qui définit la population de triggers à scorer.

In [0]:
prod_results_val_prototype = spark.read.parquet(
    f"s3://{BUCKET}/{PREFIX}/prod_results_val_prototype.parquet"
)
sessions_raw_val_prototype = spark.read.parquet(
    f"s3://{BUCKET}/{PREFIX}/sessions_raw_val_prototype.parquet"
)

print(f"prod_results_val_prototype: {prod_results_val_prototype.count()} triggers")
prod_results_val_prototype.printSchema()
display(prod_results_val_prototype.limit(1))

print(f"sessions_raw_val_prototype: {sessions_raw_val_prototype.count()} triggers")
display(sessions_raw_val_prototype.limit(1))

## Embeddings de nœuds, calculés une seule fois et réutilisés par tous les triggers

In [0]:
import torch
from torch_geometric.data import Data
from torch_geometric.transforms import ToSparseTensor

N = dataset.num_nodes
batch_size = 8192

adj_t = ToSparseTensor()(Data(edge_index=dataset.data.edge_index, num_nodes=N)).adj_t.to(device)
assert adj_t.layout == torch.sparse_csr, f"layout inattendu: {adj_t.layout}"

for m in model.model.backbone.modules():
    if hasattr(m, "_cached_edge_index"):
        m._cached_edge_index = None
    if hasattr(m, "_cached_adj_t"):
        m._cached_adj_t = None

torch.cuda.empty_cache()

model.eval()

# backbone identique au notebook ANN: t_forward doit y retrouver la meme valeur
if device == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
with torch.no_grad():
    x = model.model.embedder(dataset.data.x.to(device))
    x = model.model.backbone(x, adj_t)
if device == "cuda":
    torch.cuda.synchronize()
t_forward = time.perf_counter() - t0

# pas de normalisation globale ni d'index a construire ici: pendant de t_normalize + t_build_* cote ANN
t_prepare_retrieval = 0.0

print(f"x shape: {tuple(x.shape)}")
print(f"forward pass: {t_forward:.3f}s | preparation retrieval: {t_prepare_retrieval:.3f}s")

## Extraction des triggers — un seul chemin

Tous les triggers de `prod_results_val_prototype`, sans distinction entre exécutions avec positifs et exécutions full-negatives.

In [0]:
def extract_all_triggers(prod_results: DataFrame) -> list[dict]:
    """Tous les triggers de la table prod prototype.

    Remplace extract_triggers (qui partait de val_split["edge"], donc des seules executions ayant
    au moins un positif dans le top-12 prod) et extract_negative_only_triggers. Le tenseur d'edges
    n'est plus utilise ici: la population d'analyse est definie independamment de l'etiquetage.
    """
    rows = (
        prod_results
        .select("exec_code", "trigger_internal_id")
        .dropDuplicates(["exec_code"])
        .collect()
    )

    triggers = []
    skipped_no_node = 0

    for row in rows:
        trigger_internal_id = int(row["trigger_internal_id"])

        if trigger_internal_id not in node2idx:
            skipped_no_node += 1
            continue

        triggers.append({
            "exec_code": row["exec_code"],
            "trigger_internal_id": trigger_internal_id,
            "trigger_node_id": node2idx[trigger_internal_id],
        })

    print(f"triggers hors graphe (produit sans node_id): {skipped_no_node}/{len(rows)}")

    return triggers


t0 = time.perf_counter()
triggers = extract_all_triggers(prod_results_val_prototype)
t_extract_triggers = time.perf_counter() - t0

print(f"{len(triggers)} triggers a scorer")
print(triggers[0])
print(f"temps extraction triggers (spark): {t_extract_triggers:.3f}s")

In [0]:
def extract_categories(triggers: list[dict]) -> list[dict]:
    """Attache a chaque trigger sa t2s_best_fitting_category. trigger_internal_id est deja
    renseigne par extract_all_triggers, contrairement a la version non-prototype."""

    unique_internal_ids = {t["trigger_internal_id"] for t in triggers}

    df_trigger_ids = spark.createDataFrame(
        [(str(i),) for i in unique_internal_ids], schema="internalId string"
    )

    df_categories = (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(F.col("customer_short_name") == "maisons-du-monde")
        .join(F.broadcast(df_trigger_ids), on="internalId", how="left_semi")
        .select("internalId", F.col("t2s_best_fitting_category")[0].alias("category"))
        .dropDuplicates(["internalId"])
        .collect()
    )

    category_by_internal_id = {
        int(row["internalId"]): row["category"]
        for row in df_categories
        if row["category"] is not None
    }

    for trigger in triggers:
        trigger["category"] = category_by_internal_id.get(trigger["trigger_internal_id"])

    return triggers


t0 = time.perf_counter()
triggers = extract_categories(triggers)
t_extract_categories = time.perf_counter() - t0

missing = sum(1 for t in triggers if t["category"] is None)
print(f"triggers sans categorie: {missing}/{len(triggers)}")
print(triggers[0])
print(f"temps extraction categories (spark): {t_extract_categories:.3f}s")

In [0]:
def build_candidates_by_category(triggers: list[dict]) -> dict:

    unique_categories = {t["category"] for t in triggers if t["category"] is not None}

    df_category_products = (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(
            F.col("t2s_best_fitting_category")[0].isin(list(unique_categories))
            & (F.col("customer_short_name") == "maisons-du-monde")
        )
        .select("internalId", F.col("t2s_best_fitting_category")[0].alias("category"))
        .dropDuplicates(["internalId"])
        .collect()
    )

    internal_ids_by_category = {c: [] for c in unique_categories}
    for row in df_category_products:
        internal_ids_by_category[row["category"]].append(int(row["internalId"]))

    candidate_node_ids_by_category = {}
    for category, internal_ids in internal_ids_by_category.items():
        node_ids = torch.tensor([node2idx[i] for i in internal_ids if i in node2idx])
        candidate_node_ids_by_category[category] = torch.unique(node_ids)  # plusieurs internalId peuvent partager le même node_id

    return candidate_node_ids_by_category

t0 = time.perf_counter()
candidates_by_category = build_candidates_by_category(triggers)
t_pools = time.perf_counter() - t0

pool_sizes = [c.numel() for c in candidates_by_category.values()]
print(f"nombre de catégories: {len(candidates_by_category)}")
print(f"taille moyenne du pool de candidats par catégorie: {sum(pool_sizes) / len(pool_sizes):.0f}")
print(f"temps recuperation pools (spark): {t_pools:.3f}s")

## Inférence sur tous les triggers

Le modèle classe l'intégralité du pool de la catégorie du trigger, puis on garde son top 12. Ce pool est bien plus large que les 12 candidats de la prod : c'est ce qui permet au modèle de remonter un produit que la prod n'avait pas proposé du tout.

In [0]:
TOP_K = 12


def run_inference(triggers: list[dict], candidates_by_category: dict, timings=None) -> list[dict]:

    skipped_no_candidates = 0
    t_score = 0.0
    t_materialize = 0.0

    with torch.no_grad():
        for trigger in triggers:
            if trigger["category"] is None:
                skipped_no_candidates += 1
                continue

            # t_score s'arrete la ou s'arrete run_retrieval cote ANN: pool filtre, scoring, top-k,
            # k ids et scores rapatries. Les .cpu() forcent la synchro dans la fenetre mesuree.
            t0 = time.perf_counter()

            candidates_to_score = candidates_by_category[trigger["category"]]
            candidates_to_score = candidates_to_score[candidates_to_score != trigger["trigger_node_id"]]

            if candidates_to_score.numel() == 0:
                skipped_no_candidates += 1
                continue

            target_edges = torch.stack([
                torch.full_like(candidates_to_score, trigger["trigger_node_id"]),
                candidates_to_score,
            ])

            logits_chunks = []
            for start in range(0, target_edges.shape[1], batch_size):
                chunk = target_edges[:, start:start + batch_size].to(device)
                logits_chunks.append(model.model.classifier(x, x, chunk))

            # topk sur le device: seuls TOP_K logits et indices traversent le lien GPU -> CPU, la ou
            # un argsort complet suivi de trois .tolist() rapatriait et allouait candidate_pool_size
            # elements Python par trigger. Possible uniquement depuis que positive_ranks_model a
            # disparu: plus personne n'a besoin du classement complet, seuls les 12 premiers.
            logits = torch.cat(logits_chunks)
            k = min(TOP_K, logits.numel())  # un pool peut compter moins de 12 produits
            top_logits, top_positions = torch.topk(logits, k)
            top_node_ids = candidates_to_score[top_positions.cpu()]

            node_ids_list = top_node_ids.tolist()
            scores_list = top_logits.cpu().tolist()

            t_score += time.perf_counter() - t0
            t0 = time.perf_counter()

            top12_model = [
                {
                    "internalId": int(idx2node[node_id]),
                    "score": float(score),
                    "rank": rank,
                }
                for rank, (node_id, score) in enumerate(zip(node_ids_list, scores_list), start=1)
            ]

            t_materialize += time.perf_counter() - t0

            trigger["candidate_pool_size"] = candidates_to_score.numel()
            trigger["top12_model"] = top12_model

    if timings is not None:
        timings["score"] = t_score
        timings["materialize"] = t_materialize

    print(f"triggers non scores (categorie manquante ou pool vide): {skipped_no_candidates}/{len(triggers)}")

    return triggers

In [0]:
import copy
from torch.profiler import profile, ProfilerActivity

SAMPLE_SIZE = 20

activities = [ProfilerActivity.CPU]
if device == "cuda":
    activities.append(ProfilerActivity.CUDA)

triggers_sample = copy.deepcopy(triggers[:SAMPLE_SIZE])

run_inference(triggers_sample, candidates_by_category)
if device == "cuda":
    torch.cuda.synchronize()

triggers_sample = copy.deepcopy(triggers[:SAMPLE_SIZE])

with profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
) as prof:
    run_inference(triggers_sample, candidates_by_category)

if device == "cuda":
    torch.cuda.synchronize()
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=20))

In [0]:
# pendant MLP de la cellule time_search / time_run_retrieval du notebook ANN:
# meme categorie (plus gros pool), memes 200 requetes, meme protocole warmup + n_repeats

def time_score_mlp(trigger_node_ids, pool, k=TOP_K, n_repeats=5):

    def one_pass():
        for node_id in trigger_node_ids:
            candidates = pool[pool != node_id]
            target_edges = torch.stack([torch.full_like(candidates, node_id), candidates])

            logits_chunks = []
            for start in range(0, target_edges.shape[1], batch_size):
                chunk = target_edges[:, start:start + batch_size].to(device)
                logits_chunks.append(model.model.classifier(x, x, chunk))

            logits = torch.cat(logits_chunks)
            top_logits, top_positions = torch.topk(logits, min(k, logits.numel()))
            candidates[top_positions.cpu()].tolist()
            top_logits.cpu().tolist()

    with torch.no_grad():
        one_pass()  # warmup
        if device == "cuda":
            torch.cuda.synchronize()

        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            one_pass()
            if device == "cuda":
                torch.cuda.synchronize()  # sinon on mesure l'enqueue, pas l'execution
            times.append(time.perf_counter() - t0)

    return times


cat_test = max(candidates_by_category, key=lambda c: candidates_by_category[c].numel())
pool_test = candidates_by_category[cat_test]
qids_test = [t["trigger_node_id"] for t in triggers if t["category"] == cat_test][:200]

print(f"categorie test: {cat_test} | pool={pool_test.numel()} | requetes={len(qids_test)}")
print(f"device: {device} | torch threads: {torch.get_num_threads()}")

# 1 requete d'abord (6 passes): donne le cout unitaire avant d'engager le batch
t_mlp_single = time_score_mlp(qids_test[:1], pool_test)
per_query = np.mean(t_mlp_single)
print(f"scoring 1 requete    | {per_query * 1000:.3f}ms")

# estimations optimistes: cat_test est le plus gros pool, donc le mieux utilise sur GPU
edges_total = sum(
    candidates_by_category[t["category"]].numel() for t in triggers if t["category"] is not None
)
print(f"estimation cette cellule       | ~{per_query * len(qids_test) * 6:.1f}s (200 requetes x 6 passes)")
print(f"estimation inference complete  | ~{per_query / pool_test.numel() * edges_total / 60:.1f} min "
      f"({edges_total / 1e6:.0f}M paires)")

t_mlp_batch = time_score_mlp(qids_test, pool_test)
print(f"scoring {len(qids_test)} requetes | {np.mean(t_mlp_batch) * 1000:.3f}ms "
      f"({np.mean(t_mlp_batch) / len(qids_test) * 1000:.3f}ms par requete)")

# la ligne "1 requete" du notebook ANN mesure faiss sur CPU mono-thread, celle ci-dessus mesure le
# GPU: passer a True pour comparer a materiel egal (classifier et x bascules sur CPU puis restaures)
RUN_CPU_PARITY = False

if RUN_CPU_PARITY:
    x_gpu, device_gpu = x, device
    try:
        model.model.classifier.to("cpu")
        x, device = x.cpu(), "cpu"
        t_mlp_single_cpu = time_score_mlp(qids_test[:1], pool_test, n_repeats=3)
        print(f"scoring 1 requete (cpu, {torch.get_num_threads()} threads) | "
              f"{np.mean(t_mlp_single_cpu) * 1000:.3f}ms")
    finally:
        x, device = x_gpu, device_gpu
        model.model.classifier.to(device)

In [0]:
# meme chose, mais tous les triggers partagent les memes chunks de batch_size au lieu d'un
# forward par trigger: seul le topk final reste fait par trigger (split par taille de pool)

def time_score_mlp_batched(trigger_node_ids, pool, k=TOP_K, n_repeats=5):

    def one_pass():
        pools = [pool[pool != node_id] for node_id in trigger_node_ids]
        sizes = [p.numel() for p in pools]
        queries = torch.cat([torch.full_like(p, node_id) for node_id, p in zip(trigger_node_ids, pools)])
        candidates = torch.cat(pools)
        target_edges = torch.stack([queries, candidates])

        logits_chunks = []
        for start in range(0, target_edges.shape[1], batch_size):
            chunk = target_edges[:, start:start + batch_size].to(device)
            logits_chunks.append(model.model.classifier(x, x, chunk))
        logits = torch.cat(logits_chunks)

        for logits_q, candidates_q in zip(logits.split(sizes), candidates.split(sizes)):
            top_logits, top_positions = torch.topk(logits_q, min(k, logits_q.numel()))
            candidates_q[top_positions.cpu()].tolist()
            top_logits.cpu().tolist()

    with torch.no_grad():
        one_pass()  # warmup
        if device == "cuda":
            torch.cuda.synchronize()

        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            one_pass()
            if device == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)

    return times


t_mlp_batched = time_score_mlp_batched(qids_test, pool_test)
print(f"scoring {len(qids_test)} requetes (batch inter-triggers) | {np.mean(t_mlp_batched) * 1000:.3f}ms "
      f"({np.mean(t_mlp_batched) / len(qids_test) * 1000:.3f}ms par requete)")

In [0]:
def time_score_cosine(trigger_node_ids, pool, k=TOP_K, n_repeats=5):

    h = model.model.classifier.project(x)  # projection nodale: un seul cout, hors boucle triggers
    pool_h = h[pool]

    def one_pass():
        for node_id in trigger_node_ids:
            mask = pool != node_id
            scores = pool_h[mask] @ h[node_id]  # un seul matmul: tout le pool d'un coup, pas de chunk

            top_scores, top_positions = torch.topk(scores, min(k, scores.numel()))
            pool[mask][top_positions.cpu()].tolist()
            top_scores.cpu().tolist()

    with torch.no_grad():
        one_pass()  # warmup
        if device == "cuda":
            torch.cuda.synchronize()

        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            one_pass()
            if device == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)

    return times


t_cosine_seq = time_score_cosine(qids_test, pool_test)
print(f"scoring {len(qids_test)} requetes (cosine, sequentiel, candidats batches) | "
      f"{np.mean(t_cosine_seq) * 1000:.3f}ms "
      f"({np.mean(t_cosine_seq) / len(qids_test) * 1000:.3f}ms par requete)")

In [0]:
def time_score_cosine_batched(trigger_node_ids, pool, k=TOP_K, n_repeats=5):

    h = model.model.classifier.project(x)
    pool_device = pool.to(device)
    pool_h = h[pool_device]
    query_ids = torch.tensor(trigger_node_ids, device=device)

    def one_pass():
        scores = h[query_ids] @ pool_h.T  # un seul matmul pour tout le lot
        scores = scores.masked_fill(pool_device.unsqueeze(0) == query_ids.unsqueeze(1), float("-inf"))

        top_scores, top_positions = torch.topk(scores, k, dim=1)
        pool[top_positions.cpu()].tolist()
        top_scores.cpu().tolist()

    with torch.no_grad():
        one_pass()
        if device == "cuda":
            torch.cuda.synchronize()

        times = []
        for _ in range(n_repeats):
            t0 = time.perf_counter()
            one_pass()
            if device == "cuda":
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)

    return times


t_cosine_batched = time_score_cosine_batched(qids_test, pool_test)
print(f"scoring {len(qids_test)} requetes (cosine, batche) | "
      f"{np.mean(t_cosine_batched) * 1000:.3f}ms "
      f"({np.mean(t_cosine_batched) / len(qids_test) * 1000:.3f}ms par requete)")

In [0]:
# warmup identique cote ANN: une requete par categorie, pour ne pas mesurer un cache froid
seen_categories = set()
warmup_triggers = []
for trigger in triggers:
    if trigger["category"] is not None and trigger["category"] not in seen_categories:
        seen_categories.add(trigger["category"])
        warmup_triggers.append(copy.deepcopy(trigger))  # ne pas ecrire top12_model dans les vrais triggers

run_inference(warmup_triggers, candidates_by_category)
if device == "cuda":
    torch.cuda.synchronize()

timings = {}
t0 = time.perf_counter()
triggers = run_inference(triggers, candidates_by_category, timings=timings)
if device == "cuda":
    torch.cuda.synchronize()
t_inference = time.perf_counter() - t0

n_scored = sum(1 for t in triggers if t.get("top12_model"))
print(f"{n_scored}/{len(triggers)} triggers scores")
print(triggers[0].get("top12_model", [])[:3])
print(f"temps run_inference (toutes categories): {t_inference:.3f}s ({len(triggers) / t_inference:.0f} triggers/s)")
print(f"  dont scoring (pool + classifier + topk + rapatriement): {timings['score']:.3f}s")
print(f"  dont materialisation des dicts top12_model:             {timings['materialize']:.3f}s")
print(f"  reste (boucle python, triggers sans categorie):         {t_inference - timings['score'] - timings['materialize']:.3f}s")

## Résultats du modèle

Même structure que les variantes prod : `products_returned`, `positives`, `negatives`. Les positifs sont les produits du top-12 modèle effectivement vus dans la session — exactement la définition utilisée pour `positives_display` et `positives_relevance`, ce qui rend les trois colonnes directement comparables dans `model_stat_prototype`.

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, DoubleType

# Lignes plates loties, plutot qu'un ArrayType(StructType) construit cote Python. Trois raisons:
#   - un tuple de 4 primitifs pese ~3x moins que les 13 dicts imbriques par trigger de l'ancienne
#     version, qui constituait en plus une seconde copie complete pendant que triggers vivait encore
#   - verifySchema=False supprime la validation ligne par ligne d'un schema imbrique
#   - le lotissement rend la memoire de pointe independante du nombre de triggers, ce qui compte sur
#     un cluster single node ou les 32 Go sont partages entre REPL Python, JVM driver et executeur
# C'est Spark qui reconstruit ensuite le tableau imbrique, via collect_list + sort_array.
MODEL_ROWS_PATH = f"s3://{BUCKET}/{PREFIX}/_staging_model_rows_val_prototype.parquet"
BATCH_TRIGGERS = 20_000

model_rows_schema = StructType([
    StructField("exec_code", LongType(), True),
    StructField("internal_id", LongType(), True),
    StructField("score", DoubleType(), True),
    StructField("rank", IntegerType(), True),
])

keys_schema = StructType([
    StructField("exec_code", LongType(), True),
    StructField("trigger_internal_id", LongType(), True),
])

EMPTY_PRODUCTS = F.array().cast("array<struct<internal_id:bigint,score:double,rank:int>>")


def stage_model_rows(triggers: list[dict], path: str, batch_triggers: int = BATCH_TRIGGERS) -> None:
    """Ecrit les lignes plates par lots. Seul le premier lot ecrase, les suivants ajoutent."""
    mode = "overwrite"

    for start in range(0, len(triggers), batch_triggers):
        rows = [
            (t["exec_code"], v["internalId"], v["score"], v["rank"])
            for t in triggers[start:start + batch_triggers]
            for v in t.get("top12_model", [])
        ]
        if not rows:
            continue

        (
            spark.createDataFrame(rows, schema=model_rows_schema, verifySchema=False)
            .write.mode(mode).parquet(path)
        )
        print(f"  lot {start}-{min(start + batch_triggers, len(triggers))}: {len(rows)} lignes")
        mode = "append"

    if mode == "overwrite":  # aucun trigger scoré: on ecrit quand meme le schema
        spark.createDataFrame([], schema=model_rows_schema).write.mode("overwrite").parquet(path)
        print("  aucun trigger score, parquet vide ecrit")


def build_model_results(triggers: list[dict], sessions_raw: DataFrame) -> DataFrame:

    stage_model_rows(triggers, MODEL_ROWS_PATH)

    # Les cles portent TOUS les triggers, y compris ceux sans categorie ou a pool vide: ils gardent
    # ainsi une ligne avec products_returned vide, comme dans la version precedente.
    df_keys = spark.createDataFrame(
        [(t["exec_code"], t["trigger_internal_id"]) for t in triggers],
        schema=keys_schema,
        verifySchema=False,
    )

    # sort_array trie sur le premier champ du struct, donc rank. Le transform remet ensuite les
    # champs dans l'ordre internal_id, score, rank -- ordre auquel model_stat_prototype se fie via
    # son cast array<struct<internal_id:bigint,score:double,rank:int>>. Ne pas l'inverser.
    df_products = (
        spark.read.parquet(MODEL_ROWS_PATH)
        .groupBy("exec_code")
        .agg(F.sort_array(F.collect_list(F.struct("rank", "internal_id", "score"))).alias("ranked"))
        .withColumn(
            "products_returned",
            F.transform(
                "ranked",
                lambda r: F.struct(
                    r["internal_id"].alias("internal_id"),
                    r["score"].alias("score"),
                    r["rank"].alias("rank"),
                ),
            ),
        )
        .select("exec_code", "products_returned")
    )

    session_ids_by_exec_code = (
        sessions_raw
        .select("exec_code", F.col("session_products.internal_id").alias("session_ids"))
        .dropDuplicates(["exec_code"])
    )

    return (
        df_keys
        .join(df_products, on="exec_code", how="left")
        .join(session_ids_by_exec_code, on="exec_code", how="left")
        .withColumn("products_returned", F.coalesce(F.col("products_returned"), EMPTY_PRODUCTS))
        .withColumn("session_ids", F.coalesce(F.col("session_ids"), F.array().cast("array<bigint>")))
        .withColumn(
            "positives",
            F.filter("products_returned", lambda p: F.array_contains(F.col("session_ids"), p["internal_id"])),
        )
        .withColumn(
            "negatives",
            F.filter("products_returned", lambda p: ~F.array_contains(F.col("session_ids"), p["internal_id"])),
        )
        .select("exec_code", "trigger_internal_id", "positives", "negatives", "products_returned")
    )


# le .count() est dans la fenetre: sans action, le staging seul serait mesure (idem cote ANN)
t0 = time.perf_counter()
model_results_val_prototype = build_model_results(triggers, sessions_raw_val_prototype).cache()

print(f"model_results_val_prototype: {model_results_val_prototype.count()} triggers")
t_build_results = time.perf_counter() - t0

model_results_val_prototype.printSchema()

display(model_results_val_prototype.limit(1))
print(f"temps staging + jointures spark: {t_build_results:.3f}s")

In [0]:
t0 = time.perf_counter()
model_results_val_prototype.write.mode("overwrite").parquet(
    f"s3://{BUCKET}/{PREFIX}/model_results_val_prototype.parquet"
)
t_write = time.perf_counter() - t0

print("model_results_val_prototype.parquet uploaded")
print(f"temps ecriture parquet: {t_write:.3f}s")


#####

## Récapitulatif des temps

À comparer ligne par ligne avec la cellule équivalente de `model_test_prototype_ann.ipynb`. Les bornes de mesure sont identiques des deux côtés, à une exception près : le mapping `idx2node` et la construction des lignes se font ici dans `run_inference`, et côté ANN dans `stage_model_rows_ann` — donc dans `t_build_results`. Seule leur somme (`bloc comparable`) a des bornes rigoureusement identiques dans les deux notebooks ; pour opposer le scoring seul, utiliser `timings["score"]` face à `t_inference` de l'ANN.

Deux réserves valables pour toute la comparaison :

- **Le matériel diffère.** Ici la tête MLP tourne sur GPU, côté ANN faiss tourne sur CPU. C'est une comparaison de déploiements, pas d'algorithmes. `RUN_CPU_PARITY` dans la cellule de micro-benchmark donne le point à matériel égal.
- **Le coût ne scale pas pareil.** Le MLP évalue le pool entier par trigger (coût ∝ `pool_size × n_triggers`), faiss fait un top-k. Les temps ne sont pas transposables à un autre catalogue sans la taille des pools.

In [0]:
n_triggers = len(triggers)
bloc_comparable = t_inference + t_build_results
chaine_complete = (
    t_forward + t_prepare_retrieval + t_extract_triggers + t_extract_categories
    + t_pools + bloc_comparable + t_write
)

print(f"head: {type(model.model.classifier).__name__} | device: {device} | triggers: {n_triggers}")
print(f"checkpoint: {checkpoint_path}")
print()
print(f"forward GNN (embedder + backbone)      {t_forward:9.3f}s")
print(f"preparation retrieval                  {t_prepare_retrieval:9.3f}s")
print(f"extraction triggers (spark)            {t_extract_triggers:9.3f}s")
print(f"extraction categories (spark)          {t_extract_categories:9.3f}s")
print(f"pools de candidats (spark)             {t_pools:9.3f}s")
print(f"run_inference                          {t_inference:9.3f}s   ({n_triggers / t_inference:.0f} triggers/s)")
print(f"  dont scoring                         {timings['score']:9.3f}s")
print(f"  dont materialisation top12_model     {timings['materialize']:9.3f}s")
print(f"staging + jointures spark              {t_build_results:9.3f}s")
print(f"ecriture parquet                       {t_write:9.3f}s")
print()
print(f"bloc comparable (inference + staging)  {bloc_comparable:9.3f}s")
print(f"chaine complete                        {chaine_complete:9.3f}s")
print()
print(f"micro-benchmark ({cat_test}, pool={pool_test.numel()}, {len(qids_test)} requetes)")
print(f"  scoring batch                        {np.mean(t_mlp_batch) * 1000:9.3f}ms")
print(f"  scoring 1 requete                    {np.mean(t_mlp_single) * 1000:9.3f}ms")

In [0]:
import gc

# La fusion des deux passes de model_test a fait disparaitre le del/gc.collect() qui separait les
# deux populations, donc la pointe memoire est passee du max des deux a leur somme. On la restaure.
# Un trigger est conserve pour le controle visuel ci-dessous, avant de liberer la liste complete.
sample_trigger = next(t for t in triggers if t.get("top12_model"))

del triggers, candidates_by_category
gc.collect()

print("triggers et candidates_by_category liberes")
print(f"trigger conserve pour le controle visuel: exec_code={sample_trigger['exec_code']}")

## Contrôle visuel sur un trigger

Les trois retours côte à côte pour une même exécution : le top-12 prod dans l'ordre d'affichage, le top-12 prod par `relevanceScore`, et le top-12 du modèle. Les deux variantes prod sont lues depuis le parquet, donc elles reflètent exactement ce qui a servi aux statistiques — plus de recalcul divergent depuis `ads_adlog_fct`.

In [0]:
def get_customer_db_name(customer_shortname: str) -> str:
    df_customer = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_gold_customer")
        .where(F.col("shortName") == customer_shortname)
        .select(F.col("databaseName").alias("db_name"))
    )
    return df_customer.collect()[0]["db_name"]


def display_product_model(internal_ids: list[int], db_name: str, image_width: int = 150) -> None:
    if not internal_ids:
        print("(aucun produit)")
        return

    ranks_df = spark.createDataFrame(
        [Row(internalId=i, topk_rank=rank) for rank, i in enumerate(internal_ids, start=1)]
    )

    df_products = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_mongo_product_0_current")
        .filter(F.col("db_name") == db_name)
        .select(F.col("internalId").cast("bigint").alias("internalId"), "name", "imageUrl")
        .dropDuplicates(["internalId"])
    )

    df_products_info = (
        ranks_df
        .join(df_products, on="internalId", how="left")
        .select("topk_rank", "internalId", "name", "imageUrl")
        .orderBy("topk_rank")
    )

    for row in df_products_info.collect():
        if row["imageUrl"] is None:
            print(f"{row['topk_rank']}. internalId={row['internalId']} — pas d'image trouvee")
            continue
        display(HTML(f"<p><b>{row['topk_rank']}.</b> {row['name']} (internalId={row['internalId']})</p>"))
        display(HTML(f'<img src="{row["imageUrl"]}" width="{image_width}">'))


db_name = get_customer_db_name("maisons-du-monde")

In [0]:
# sample_trigger vient de la cellule de liberation memoire ci-dessus
sample_exec_code = sample_trigger["exec_code"]

sample_prod = (
    prod_results_val_prototype
    .filter(F.col("exec_code") == sample_exec_code)
    .collect()[0]
)


def ids_in_rank_order(products) -> list[int]:
    return [int(p["internal_id"]) for p in sorted(products, key=lambda p: p["rank"])]


prod_display_ids = ids_in_rank_order(sample_prod["products_returned_display"])
prod_relevance_ids = ids_in_rank_order(sample_prod["products_returned_relevance"])
model_ids = [int(p["internalId"]) for p in sample_trigger["top12_model"]]

print(f"exec_code {sample_exec_code} -> executionId prod: {sample_prod['execution_id']}")
print(f"produit declencheur: internalId={sample_trigger['trigger_internal_id']}")
print(f"categorie: {sample_trigger['category']} | pool de candidats: {sample_trigger['candidate_pool_size']}")
print()
print(f"prod display  : {prod_display_ids}")
print(f"prod relevance: {prod_relevance_ids}")
print(f"modele        : {model_ids}")
print()
print(f"overlap display/relevance : {len(set(prod_display_ids) & set(prod_relevance_ids))}")
print(f"overlap modele/display    : {len(set(model_ids) & set(prod_display_ids))}")
print(f"overlap modele/relevance  : {len(set(model_ids) & set(prod_relevance_ids))}")

In [0]:
for title, ids in [
    ("Prod — 12 premiers dans l'ordre d'affichage", prod_display_ids),
    ("Prod — 12 premiers par relevanceScore", prod_relevance_ids),
    ("Modele — top 12", model_ids),
]:
    display(HTML(f"<h3>{title}</h3>"))
    display_product_model(ids, db_name=db_name)